In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Generate synthetic X
num_subjects = 500
num_features = 5

X_raw = torch.randn(num_subjects, num_features)


# 2. Add intercept column
ones_column = torch.ones(num_subjects, 1)

X = torch.cat((ones_column, X_raw), dim=1)
# X shape: (500, 6)


# 3. Define true beta
# 第一個是 intercept
beta_true = torch.tensor([
    1.0,   # intercept
    2.0,   # X1 effect
    -1.5,  # X2 effect
    0.5,   # X3 effect
    0.0,   # X4 no effect
    3.0    # X5 effect
]).reshape(-1,1)


# 4. Generate Y according to linear model
noise = torch.randn(num_subjects,1) * 0.5

Y_raw = X @ beta_true + noise

In [2]:
# 確認模擬資料是對的
beta = torch.linalg.solve(
    X.T @ X,
    X.T @ Y_raw
)

print(beta)

tensor([[ 1.0032],
        [ 2.0408],
        [-1.5013],
        [ 0.5562],
        [-0.0200],
        [ 3.0278]])


In [3]:
from sklearn.model_selection import train_test_split

# 建立 index
indices = torch.arange(num_subjects)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

# 切資料
X_train_raw = X_raw[train_idx]
X_test_raw = X_raw[test_idx]

Y_train = Y_raw[train_idx]
Y_test = Y_raw[test_idx]

In [4]:
# Train跟test都要加截距項
ones_train = torch.ones(X_train_raw.shape[0],1)
ones_test = torch.ones(X_test_raw.shape[0],1)


X_train = torch.cat(
    (ones_train, X_train_raw),
    dim=1
)

X_test = torch.cat(
    (ones_test, X_test_raw),
    dim=1
)

In [5]:
# 先把Y也加入X中，要讓attention matrix有Y的資訊
X_Y_train = torch.cat(
    (X_train, Y_train),
    dim=1
)

In [6]:
# 原始程式碼貼上(不訓練attention)

# Transpose data so the 6 features plus Y act as the "sequence" 
X_Y_features = X_Y_train.t()

# 5. Define the projection dimension (d_k)
d_k = 32
W_Q = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

W_K = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

# 6. Project features into Query (Q) and Key (K) spaces
Q = W_Q(X_Y_features)
K = W_K(X_Y_features)

# 7. Compute the raw attention scores
scores = torch.matmul(
    Q,
    K.transpose(-2,-1)
)

# 8. Scale by sqrt(d_k) and apply Softmax row-wise
attention_matrix = F.softmax(
    scores / (d_k ** 0.5),
    dim=-1
)

In [7]:
print("Attention Matrix Shape:", attention_matrix.shape)
print("\nAttention Matrix:\n", attention_matrix)

Attention Matrix Shape: torch.Size([7, 7])

Attention Matrix:
 tensor([[0.1919, 0.1463, 0.0958, 0.1782, 0.0904, 0.0982, 0.1992],
        [0.1189, 0.1435, 0.1106, 0.1826, 0.2229, 0.1059, 0.1155],
        [0.1939, 0.0878, 0.2042, 0.1139, 0.0740, 0.1967, 0.1296],
        [0.1885, 0.1253, 0.1813, 0.1699, 0.1229, 0.1613, 0.0508],
        [0.0957, 0.1838, 0.1560, 0.0964, 0.2131, 0.1445, 0.1105],
        [0.1071, 0.1234, 0.1186, 0.0583, 0.0619, 0.1154, 0.4153],
        [0.0183, 0.0617, 0.0093, 0.0100, 0.0145, 0.0074, 0.8789]],
       grad_fn=<SoftmaxBackward0>)


In [8]:
# 把Y再從矩陣中拿掉
A = attention_matrix[:-1,:-1]

print("Attention Matrix Shape(拿掉Y):", A.shape)
print("\nAttention Matrix(拿掉Y):\n", A)

Attention Matrix Shape(拿掉Y): torch.Size([6, 6])

Attention Matrix(拿掉Y):
 tensor([[0.1919, 0.1463, 0.0958, 0.1782, 0.0904, 0.0982],
        [0.1189, 0.1435, 0.1106, 0.1826, 0.2229, 0.1059],
        [0.1939, 0.0878, 0.2042, 0.1139, 0.0740, 0.1967],
        [0.1885, 0.1253, 0.1813, 0.1699, 0.1229, 0.1613],
        [0.0957, 0.1838, 0.1560, 0.0964, 0.2131, 0.1445],
        [0.1071, 0.1234, 0.1186, 0.0583, 0.0619, 0.1154]],
       grad_fn=<SliceBackward0>)


In [9]:
# 矩陣乘上Y的變異數
var_y = torch.var(Y_train)

A_var_y = A * var_y

In [10]:
A_var_y

tensor([[2.9272, 2.2317, 1.4607, 2.7185, 1.3787, 1.4970],
        [1.8141, 2.1894, 1.6866, 2.7847, 3.3998, 1.6154],
        [2.9571, 1.3387, 3.1141, 1.7373, 1.1280, 3.0004],
        [2.8746, 1.9108, 2.7653, 2.5918, 1.8737, 2.4609],
        [1.4601, 2.8032, 2.3797, 1.4698, 3.2496, 2.2037],
        [1.6337, 1.8821, 1.8084, 0.8891, 0.9446, 1.7606]],
       grad_fn=<MulBackward0>)

In [11]:
X_train.T @ X_train

tensor([[400.0000, -11.5514, -19.6436,  -2.2583, -20.4130, -31.1513],
        [-11.5514, 378.2653, -13.3198,   1.0643, -20.1379, -30.8461],
        [-19.6436, -13.3198, 450.3373,  19.9203, -26.0783, -11.5691],
        [ -2.2583,   1.0643,  19.9203, 418.4709,  47.1536, -57.4135],
        [-20.4130, -20.1379, -26.0783,  47.1536, 410.7896, -16.9338],
        [-31.1513, -30.8461, -11.5691, -57.4135, -16.9338, 405.0438]])

In [12]:
# 放入OLS的公式解中，估計beta
beta_attention = torch.linalg.solve(
    (X_train.T @ X_train + A_var_y)/2,
    X_train.T @ Y_train
)

In [13]:
# 把算出來的係數套入驗證集
Y_pred_attention = X_test @ beta_attention

In [14]:
# 算MSE
mse_attention = torch.mean(
    (Y_test - Y_pred_attention)**2
)

print(
    "Attention MSE:",
    mse_attention.item()
)

Attention MSE: 21.959535598754883


In [15]:
# OLS的標準解法
beta_ols = torch.linalg.solve(
    X_train.T @ X_train,
    X_train.T @ Y_train
)

Y_pred_ols = X_test @ beta_ols

mse_ols = torch.mean(
    (Y_test - Y_pred_ols)**2
)

print(
    "OLS MSE:",
    mse_ols.item()
)

OLS MSE: 0.22753161191940308
